# Crop Negative Control Experiments

Ce notebook reprend le script `crop_negative_control_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Controle negatif des labels crop avant de les utiliser dans une politique live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Crop attention/PPE label-shuffle negative control.
- Artefacts controles : Crop label-shuffle negative control exists. (`runs/exp_026_crop_negative_control/metrics/crop_negative_control_summary.csv`).
- Run par defaut : `runs/exp_026_crop_negative_control`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "crop_negative_control_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader

from crop_cnn_experiments import CropDataset, TARGETS, predict, train_crop_model, video_metrics
from ml_pipeline import ROOT, safe_auc, write_json
from sequence_experiments import append_report, make_run_dir


## Fonction `load_source_index`

Cette cellule definit `load_source_index`. Elle prepare une partie du script.

In [ ]:
def load_source_index(crop_run):
    source = Path(crop_run)
    if not source.is_absolute():
        source = ROOT / source
    index = pd.read_csv(source / "features" / "crop_cnn_index.csv")
    # Reuse crop images from the source run. CropDataset joins run_dir/path;
    # absolute paths survive Path joining on Windows.
    index["path"] = index["path"].apply(lambda p: str(source / p))
    return source, index


## Fonction `parent_label_shuffle`

Cette cellule definit `parent_label_shuffle`. Elle prepare une partie du script.

In [ ]:
def parent_label_shuffle(index, target, seed):
    rng = np.random.default_rng(seed)
    out = index.copy()
    label_col = f"{target}_label"
    manifest = []
    for split, split_df in index.groupby("split", sort=True):
        videos = (
            split_df.groupby("video_id", as_index=False)
            .agg(original_label=(label_col, "max"), crop_rows=(label_col, "size"))
            .sort_values("video_id")
            .reset_index(drop=True)
        )
        labels = videos["original_label"].to_numpy(dtype=int).copy()
        shuffled = labels.copy()
        if len(shuffled) > 1 and len(np.unique(shuffled)) > 1:
            for _ in range(50):
                rng.shuffle(shuffled)
                if np.any(shuffled != labels):
                    break
        label_by_video = dict(zip(videos["video_id"], shuffled))
        out.loc[out["split"].eq(split), label_col] = out.loc[out["split"].eq(split), "video_id"].map(label_by_video).astype(int)
        for _, row in videos.iterrows():
            manifest.append(
                {
                    "target": target,
                    "split": split,
                    "video_id": row["video_id"],
                    "original_label": int(row["original_label"]),
                    "control_label": int(label_by_video[row["video_id"]]),
                    "crop_rows": int(row["crop_rows"]),
                    "shuffle_seed": int(seed),
                }
            )
    return out, pd.DataFrame(manifest)


## Fonction `crop_level_metrics`

Cette cellule definit `crop_level_metrics`. Elle prepare une partie du script.

In [ ]:
def crop_level_metrics(pred_df, target):
    rows = []
    for split, group in pred_df.groupby("split"):
        y = group[f"{target}_label"].astype(int).to_numpy()
        p = group["risk"].to_numpy()
        best = None
        for threshold in np.arange(0.05, 1.0, 0.05):
            pred = (p >= threshold).astype(int)
            row = {
                "threshold": float(round(threshold, 2)),
                "f1": float(f1_score(y, pred, zero_division=0)),
                "accuracy": float(accuracy_score(y, pred)),
                "balanced_accuracy": float(balanced_accuracy_score(y, pred)) if len(np.unique(y)) > 1 else None,
                "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist(),
            }
            if best is None or row["f1"] > best["f1"]:
                best = row
        rows.append(
            {
                "split": split,
                "n_samples": int(len(group)),
                "positive_samples": int(y.sum()),
                "prevalence": float(y.mean()) if len(y) else 0.0,
                "average_precision": safe_auc(average_precision_score, y, p),
                "roc_auc": safe_auc(roc_auc_score, y, p),
                **best,
            }
        )
    return rows


## Fonction `evaluate_predictions`

Cette cellule definit `evaluate_predictions`. Elle prepare une partie du script.

In [ ]:
def evaluate_predictions(index_for_labels, probs, target, label_source, architecture, seed, level):
    pred = index_for_labels[["video_id", "split", "frame", "time_s", f"{target}_label"]].copy()
    pred["target"] = target
    pred["architecture"] = architecture
    pred["repeat_seed"] = int(seed)
    pred["risk"] = probs
    metric_rows = crop_level_metrics(pred, target) if level == "crop" else video_metrics(pred, target)
    rows = []
    for row in metric_rows:
        out = dict(row)
        out.update(
            {
                "target": target,
                "architecture": architecture,
                "repeat_seed": int(seed),
                "label_source": label_source,
                "level": level,
            }
        )
        if level == "video":
            out["prevalence"] = out["positive_videos"] / max(1, out["n_videos"])
        rows.append(out)
    return pred, rows


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(metrics, run_dir):
    rows = []
    for (target, architecture, label_source, level, split), group in metrics.groupby(
        ["target", "architecture", "label_source", "level", "split"]
    ):
        rows.append(
            {
                "target": target,
                "architecture": architecture,
                "label_source": label_source,
                "level": level,
                "split": split,
                "runs": int(group["repeat_seed"].nunique()),
                "prevalence_mean": float(group["prevalence"].mean()),
                "ap_mean": float(group["average_precision"].mean()),
                "ap_std": float(group["average_precision"].std(ddof=0)),
                "roc_auc_mean": float(group["roc_auc"].mean()) if group["roc_auc"].notna().any() else None,
                "f1_mean": float(group["f1"].mean()),
                "balanced_accuracy_mean": float(group["balanced_accuracy"].mean()) if group["balanced_accuracy"].notna().any() else None,
            }
        )
    summary = pd.DataFrame(rows)
    summary.to_csv(run_dir / "metrics" / "crop_negative_control_summary.csv", index=False)

    control_test = summary[
        (summary["label_source"] == "control_shuffled") & (summary["split"] == "test")
    ].sort_values(["target", "level", "ap_mean"], ascending=[True, True, False])
    real_test = summary[
        (summary["label_source"] == "real_original") & (summary["split"] == "test")
    ].sort_values(["target", "level", "ap_mean"], ascending=[True, True, False])

    lines = ["# Crop Label-Shuffle Negative Control", ""]
    lines.append("Parent-video labels are shuffled inside each split for each target. All crops from a parent receive the same shuffled label. This preserves video-level positive counts per split while breaking the real visual-label relationship.")
    lines.append("")
    lines.append("## Test Metrics Against Shuffled Control Labels")
    lines.append("")
    lines.append("| target | level | architecture | runs | prevalence | AP mean | AP std | ROC AUC mean | F1 mean |")
    lines.append("|---|---|---|---:|---:|---:|---:|---:|---:|")
    for _, row in control_test.iterrows():
        auc = row["roc_auc_mean"] if pd.notna(row["roc_auc_mean"]) else "NA"
        auc_text = f"{auc:.3f}" if isinstance(auc, float) else str(auc)
        lines.append(
            f"| {row['target']} | {row['level']} | {row['architecture']} | {int(row['runs'])} | {row['prevalence_mean']:.3f} | {row['ap_mean']:.3f} | {row['ap_std']:.3f} | {auc_text} | {row['f1_mean']:.3f} |"
        )
    lines.append("")
    lines.append("## Same Models Scored Against Real Labels")
    lines.append("")
    lines.append("| target | level | architecture | runs | prevalence | AP mean | AP std | ROC AUC mean | F1 mean |")
    lines.append("|---|---|---|---:|---:|---:|---:|---:|---:|")
    for _, row in real_test.iterrows():
        auc = row["roc_auc_mean"] if pd.notna(row["roc_auc_mean"]) else "NA"
        auc_text = f"{auc:.3f}" if isinstance(auc, float) else str(auc)
        lines.append(
            f"| {row['target']} | {row['level']} | {row['architecture']} | {int(row['runs'])} | {row['prevalence_mean']:.3f} | {row['ap_mean']:.3f} | {row['ap_std']:.3f} | {auc_text} | {row['f1_mean']:.3f} |"
        )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- Shuffled-label performance should fall near prevalence/chance.")
    lines.append("- Because the test set has only 11 parent videos, video-level AP is noisy. Crop-level AP is the more stable sanity check, while video-level AP remains closer to the real reporting target.")
    lines.append("- This control does not prove actor/background robustness; it only checks for basic leakage/metric failure in the crop classifier pipeline.")
    (run_dir / "crop_negative_control_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return summary


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    source, original_index = load_source_index(args.crop_run)
    run_dir = make_run_dir(args.run_name)
    write_json(
        run_dir / "config.json",
        {
            "crop_run": str(source),
            "seeds": args.seeds,
            "architectures": args.architectures,
            "targets": args.targets,
            "epochs": args.epochs,
            "patience": args.patience,
            "image_size": args.image_size,
            "control": "parent-video labels shuffled inside each split",
        },
    )
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    all_metrics = []
    all_history = []
    all_manifests = []
    for seed in args.seeds:
        for target in args.targets:
            control_index, manifest = parent_label_shuffle(original_index, target, seed)
            all_manifests.append(manifest)
            control_index.to_csv(run_dir / "features" / f"control_index_{target}_seed{seed}.csv", index=False)
            for architecture in args.architectures:
                print(f"training control seed{seed} {target} {architecture}")
                model_args = SimpleNamespace(
                    image_size=args.image_size,
                    batch_size=args.batch_size,
                    lr=args.lr,
                    weight_decay=args.weight_decay,
                    epochs=args.epochs,
                    patience=args.patience,
                    no_pretrained=args.no_pretrained,
                )
                model, history, train_time_s, model_size = train_crop_model(
                    run_dir, control_index, target, architecture, model_args, device
                )
                model_path = run_dir / "models" / f"{target}_{architecture}.pt"
                renamed = run_dir / "models" / f"{target}_seed{seed}_{architecture}_negative_control.pt"
                if model_path.exists():
                    model_path.replace(renamed)
                for row in history:
                    row["repeat_seed"] = int(seed)
                    row["label_source"] = "control_shuffled"
                all_history.extend(history)
                eval_loader = DataLoader(
                    CropDataset(run_dir, control_index, target, train=False, image_size=args.image_size),
                    batch_size=args.batch_size,
                    shuffle=False,
                    num_workers=0,
                )
                probs, _ = predict(model, eval_loader, device)
                for label_source, label_index in [("control_shuffled", control_index), ("real_original", original_index)]:
                    pred, rows = evaluate_predictions(
                        label_index, probs, target, label_source, architecture, seed, "crop"
                    )
                    all_metrics.extend(rows)
                    _, rows = evaluate_predictions(
                        label_index, probs, target, label_source, architecture, seed, "video"
                    )
                    all_metrics.extend(rows)
                    pred.to_csv(
                        run_dir
                        / "features"
                        / f"predictions_{label_source}_{target}_seed{seed}_{architecture}.csv",
                        index=False,
                    )
                pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "crop_negative_control_metrics.csv", index=False)
                pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "crop_negative_control_training_history.csv", index=False)
    if all_manifests:
        pd.concat(all_manifests, ignore_index=True).to_csv(run_dir / "features" / "parent_label_shuffle_manifest.csv", index=False)
    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "crop_negative_control_metrics.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "crop_negative_control_training_history.csv", index=False)
    summarize(metrics, run_dir)
    append_report(
        run_dir,
        "Crop Negative Control Completion",
        f"- Source crop run: `{source}`\n- Seeds: `{args.seeds}`\n- Targets: `{args.targets}`\n- Architectures: `{args.architectures}`\n- Summary: `{run_dir / 'crop_negative_control_summary.md'}`",
    )
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Crop attention/PPE label-shuffle negative control.")
    parser.add_argument("--crop-run", default="runs/exp_013_crop_cnn_catalogue")
    parser.add_argument("--run-name", default="exp_026_crop_negative_control")
    parser.add_argument("--targets", nargs="+", default=["attention", "blouse"])
    parser.add_argument("--architectures", nargs="+", default=["small_cnn", "mobilenet_v3_small"])
    parser.add_argument("--seeds", nargs="+", type=int, default=[901, 902, 903])
    parser.add_argument("--image-size", type=int, default=160)
    parser.add_argument("--epochs", type=int, default=8)
    parser.add_argument("--patience", type=int, default=2)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--no-pretrained", action="store_true")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_026_crop_negative_control_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["crop_negative_control_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
